# Notebook per esportare pickle per streamlit 

Per i dataset Titanic e German vengono esportati:
* X train per le instance
* Explainer
* bbox - Il modello è un random forest
* La feature importance con Shap
* 20 istanze per dataset già spiegate


In [13]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt

import os
from plot_explanation import PlotExplanation
import pickle

In [14]:
datasets=['titanic_c.csv','german_credit.csv']

## Titanic

In [15]:
source_file = f'../datasets/{datasets[0]}'
class_field = 'Survived'
# Load and transform dataset
df_t = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [16]:
df_t, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df_t, class_field)

### Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [17]:
test_size = 0.3
random_state = 42
X_train_t, X_test_t, Y_train_t, Y_test_t = train_test_split(df_t[feature_names], df_t[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df_t[class_field])

In [18]:
bb_t = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb_t.fit(X_train_t.values, Y_train_t.values)
bbox_t = sklearn_classifier_wrapper(bb_t)

### Export Pickle for X_train and bbox

In [8]:
dataset_dict_tit = {"X_train_titanic": X_train_t, "X_test_titanic": X_test_t, "y_train_titanic": Y_train_t, "y_test_titanic": Y_test_t}
path="../datasets/titanic/train_test_titanic.pkl"
with open(path, 'wb') as train_test_titanic:
    pickle.dump(dataset_dict_tit, train_test_titanic)

In [9]:
pickle.dump(bbox_t, open('../datasets/titanic/bbox_titanic.pkl', 'wb'))

In [35]:
inst_t = X_train_t.iloc[4].values
print('Instance ',inst_t)
print('True class ',Y_train_t.iloc[4])
print('Predicted class ',bb_t.predict(inst_t.reshape(1, -1)))

Instance  [ 3.    1.   28.    1.    0.   15.85  2.  ]
True class  0
Predicted class  [0]


### Export the explainer

In [36]:
explainer_t = LoreTabularExplainer(bbox_t)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer_t.fit(df_t, class_field, config)
pickle.dump(explainer_t, open('../datasets/titanic/explainer_titanic.pkl', 'wb'))

### Export the feature Importance

In [74]:
explainer_s_t = ShapXAITabularExplainer(bbox_t, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train_t.iloc[0:].values}
explainer_s_t.fit(config)
pickle.dump(explainer_s_t, open('../datasets/titanic/shap_featureimportance_titanic.pkl', 'wb'))

In [72]:
exp_t = explainer_s_t.explain(inst_t)

In [73]:
shap_feature_importance_t=exp_t.exp

## Export the pickle with instances explained

In [33]:
list_of_inst_t=X_train_t.iloc[0:19].values

In [37]:
list_exp_t=[]
for inst_t in list_of_inst_t:
    exp = explainer_t.explain(inst_t)
    list_exp_t.append(exp)
    print(exp)

In [38]:
list_exp_t

In [39]:
pickle.dump(list_exp_t, open('../datasets/titanic/instances_explained_titanic.pkl', 'wb'))

## German

In [40]:
source_file = f'../datasets/{datasets[1]}'
class_field = 'default'
# Load and transform dataset
df_g = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [41]:
df_g, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df_g, class_field)

### Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [42]:
test_size = 0.3
random_state = 42
X_train_g, X_test_g, Y_train_g, Y_test_g= train_test_split(df_g[feature_names], df_g[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df_g[class_field])

In [43]:
bb_g = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb_g.fit(X_train_g.values, Y_train_g.values)
bbox_g = sklearn_classifier_wrapper(bb_g)

### Export Pickle for X_train and bbox

In [44]:
dataset_dict_ger = {"X_train_german": X_train_g, "X_test_german": X_test_g, "y_train_german": Y_train_g, "y_test_german": Y_test_g}
path="../datasets/german/train_test_german.pkl"
with open(path, 'wb') as train_test_german:
    pickle.dump(dataset_dict_ger, train_test_german)

In [45]:
pickle.dump(bbox_g, open('../datasets/german/bbox_german.pkl', 'wb'))

In [46]:
inst_g = X_train_g.iloc[4].values
print('Instance ',inst_g)
print('True class ',Y_train_g.iloc[4])
print('Predicted class ',bb_g.predict(inst_g.reshape(1, -1)))

Instance  [  11 7228    1    4   39    2    1    0    0    0    1    0    1    0
    0    0    0    0    1    0    0    0    0    0    0    0    0    1
    0    0    0    0    0    1    0    0    0    0    0    1    0    0
    1    1    0    0    0    0    1    0    0    1    0    0    0    0
    1    1    0    0    1]
True class  0
Predicted class  [0]


### Export the explainer

In [47]:
explainer_g = LoreTabularExplainer(bbox_g)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer_g.fit(df_g, class_field, config)
pickle.dump(explainer_g, open('../datasets/german/explainer_german.pkl', 'wb'))


In [48]:
exp = explainer_g.explain(inst_g)
print(exp)

### Export the feature Importance

In [49]:
explainer_s_g = ShapXAITabularExplainer(bbox_g, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train_g.iloc[0:].values}
explainer_s_g.fit(config)
pickle.dump(explainer_s_g, open('../datasets/german/shap_featureimportance_german.pkl', 'wb'))

In [50]:
exp_g = explainer_s_g.explain(inst_g)

In [51]:
shap_feature_importance=exp_g.exp

## Export the pickle with instances explained

In [52]:
list_of_inst_g=X_train_g.iloc[0:19].values

In [54]:
list_of_inst_g[2]

array([  18, 4165,    2,    2,   36,    2,    2,    0,    0,    0,    1,
          0,    0,    0,    0,    1,    0,    1,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    1,    0,    0,    0,    0,    0,
          1,    0,    0,    0,    0,    0,    1,    0,    0,    1,    0,
          1,    0,    0,    0,    0,    1,    0,    1,    0,    0,    1,
          0,    0,    1,    0,    0,    1])

In [55]:
list_exp_g=[]
for inst_g in list_of_inst_g:
    exp = explainer_g.explain(inst_g)
    list_exp_g.append(exp)
    print(exp)

In [56]:
list_exp_g

In [60]:
list_exp_g[1]

In [57]:
pickle.dump(list_exp_g, open('../datasets/german/instances_explained_german.pkl', 'wb'))